# AKILAN PDF Artifact Workbench

This notebook downloads the declared public PDF corpus, validates every source with PyMuPDF, builds rendered AKILAN artifacts, and persists an auditable validation report.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from akilan import ExtractionConfig, PDFArtifactBuilder, sync_corpus


In [ ]:
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'data'
ARTIFACT_DIR = REPO_ROOT / 'artifacts' / 'notebook'
MANIFEST = DATA_DIR / 'sources.json'


## Synchronize public PDFs

Existing valid files are reused. Set `overwrite=True` only when intentionally refreshing publisher content.

In [ ]:
downloads = sync_corpus(MANIFEST, DATA_DIR)
for item in downloads:
    print(item.status, item.source_id, item.page_count, item.path.relative_to(REPO_ROOT), item.sha256[:12])


In [ ]:
PDFS = sorted(DATA_DIR.rglob('*.pdf'))
print(f'Discovered {len(PDFS)} PDF(s) under {DATA_DIR}')
for pdf in PDFS:
    print('-', pdf.relative_to(REPO_ROOT))


## Build rendered artifacts

Each PDF is isolated so one difficult file cannot stop the remaining corpus. Page renders are enabled for visual inspection.

In [ ]:
config = ExtractionConfig(render_pages=True, include_characters=False, overwrite=True)
builder = PDFArtifactBuilder(config)
results = []
for pdf in PDFS:
    relative_pdf = pdf.relative_to(DATA_DIR)
    output_dir = ARTIFACT_DIR / relative_pdf.with_suffix('')
    try:
        artifact = builder.build(pdf, output_dir)
        results.append({'pdf': str(pdf.relative_to(REPO_ROOT)), 'status': 'passed', 'output': str(output_dir.relative_to(REPO_ROOT)), 'statistics': artifact.statistics, 'error': None})
    except Exception as exc:
        results.append({'pdf': str(pdf.relative_to(REPO_ROOT)), 'status': 'failed', 'output': str(output_dir.relative_to(REPO_ROOT)), 'statistics': {}, 'error': f'{type(exc).__name__}: {exc}'})
results


## Render inventory and validation report

The final cell lists generated page images and writes a machine-readable report suitable for `user_feedback.md` evidence.

In [ ]:
rendered_pages = sorted(ARTIFACT_DIR.rglob('assets/renders/*.png'))
print(f'Rendered pages: {len(rendered_pages)}')
for render in rendered_pages[:20]:
    print('-', render.relative_to(REPO_ROOT))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
report_path = ARTIFACT_DIR / 'validation_report.json'
report_path.write_text(json.dumps(results, indent=2, default=str) + '\n', encoding='utf-8')
print(f'Wrote {report_path}')
print(f'Passed: {sum(row["status"] == "passed" for row in results)}')
print(f'Failed: {sum(row["status"] == "failed" for row in results)}')
